# FDTD на GPU Colab: Compute Shader, float32

**Выберите GPU-среду Colab (например T4) и выполните все ячейки.**
Сначала проверяется настоящий NVIDIA OpenGL-контекст. Программный рендерер
`llvmpipe` не допускается: если аппаратный EGL недоступен, расчёт остановится.

Это самостоятельный учебный пример Compute Shader и исходная реализация
для будущего сравнения с Fragment Shader. Здесь реализован
**TMz FDTD**: компоненты Ez, Hx, Hy, производная гауссова импульса в центре,
поглощающий PML Беренже либо отражающие PEC-стенки. Рупор TEz пока не входит
в этот эксперимент. Существующий NumPy-блокнот остаётся отдельным примером.

Все определения и вызовы видны в ячейках. Python-файлы на VM не создаются.
NVIDIA GPU считает поля и раскрашивает изображение; CPU передаёт команды,
формирует значение источника и кодирует видео через FFmpeg.


In [ ]:
# @title 1. Библиотеки OpenGL
%pip -q install moderngl==5.12.0 glcontext==3.0.0


In [ ]:
from pathlib import Path
import ctypes.util
import json
import math
import os
import time
import numpy as np


## 2. Проверка аппаратного OpenGL

EGL создаёт контекст без окна или X-сервера. В проверенной Colab-сессии
библиотеки NVIDIA находились в `/usr/lib64-nvidia`, но стандартный EGL
выбирал Mesa/CPU. Ниже явно указывается существующая NVIDIA EGL-библиотека
через небольшой vendor JSON. Драйвер не переустанавливается.

Ожидаем `GL_VENDOR = NVIDIA Corporation`, имя GPU в `GL_RENDERER` и
OpenGL >= 4.3 нужен для compute shaders. Fragment Shader пример
использует ту же версию API. После неудачного создания Mesa-контекста
перезапустите ядро перед повторным запуском этого блока.


In [ ]:
def create_nvidia_context(directory):
    """Select the NVIDIA EGL vendor before the first EGL context is created."""
    import moderngl
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    candidates = [Path("/usr/lib64-nvidia/libEGL_nvidia.so.0"),
                  Path("/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0"),
                  Path("/usr/local/nvidia/lib64/libEGL_nvidia.so.0")]
    library = next((str(p) for p in candidates if p.exists()), None)
    library = library or ctypes.util.find_library("EGL_nvidia")
    if library:
        vendor = directory / "nvidia_egl_vendor.json"
        vendor.write_text(json.dumps({"file_format_version": "1.0.0",
                                      "ICD": {"library_path": library}}))
        os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(vendor.resolve())
    try:
        ctx = moderngl.create_context(standalone=True, backend="egl", require=430)
    except Exception as error:
        raise RuntimeError("NVIDIA EGL/OpenGL 4.3 недоступен. Проверьте GPU-среду; "
                           "после смены EGL vendor перезапустите ядро.") from error
    info = {key: ctx.info[key] for key in ("GL_VENDOR", "GL_RENDERER", "GL_VERSION")}
    if "nvidia" not in info["GL_VENDOR"].lower() or any(
            name in info["GL_RENDERER"].lower() for name in ("llvmpipe", "softpipe", "swiftshader")):
        ctx.release()
        raise RuntimeError(f"Требуется аппаратный NVIDIA OpenGL, получен {info}. "
                           "Выберите GPU и перезапустите ядро.")
    (directory / "opengl_context.json").write_text(json.dumps(info, indent=2))
    print(json.dumps(info, indent=2))
    return ctx


In [ ]:
# @title Создание NVIDIA EGL-контекста
WORK_DIR = Path.cwd() / "balanis_compute"
WORK_DIR.mkdir(exist_ok=True)
ctx = create_nvidia_context(WORK_DIR)
print("NVIDIA_EGL_OK")


## 3. Сетка, коэффициенты и небольшой CPU-эталон

Область всегда имеет размер **0,15×0,15 м**. При увеличении числа ячеек
уменьшаются `dx` и `dt`, а длительность импульса и число шагов масштабируются
так, чтобы сохранять физическую временную шкалу. Толщина PML также
масштабируется вместе с сеткой. Используется исходное возбуждение `voltage/dx`.

Все поля, текстуры, передаваемые коэффициенты и арифметика шейдеров —
**float32**. Константы и профили PML предварительно рассчитываются на CPU
и один раз преобразуются в float32. `TMReference` нужен только для тестов
на маленькой сетке и не участвует в GPU-замерах.

Массивы имеют индексы `[i, j]`, texel OpenGL — `(j, i)`.
Поля упакованы в текстуры: E=(Ez, Ezx, Ezy, 0), H=(Hx, Hy).
Для удобства они дополнены до одинакового размера (N+1)×(N+1).


In [ ]:
class TMConfig:
    """Fixed 0.15 m square; refining the grid also reduces dt and scales the pulse."""

    def __init__(self, size=1000, boundary="pml"):
        if not isinstance(size, int) or isinstance(size, bool) or size < 16:
            raise ValueError("size must be an integer >= 16")
        if boundary not in ("pml", "pec"):
            raise ValueError("boundary must be pml or pec")
        self.size = size
        self.boundary = boundary
        self.pml = max(2, round(6 * size / 50)) if boundary == "pml" else 0
        self.mu = 4 * np.pi * 1e-7
        self.eps = 8.854e-12
        self.c = 1 / np.sqrt(self.mu * self.eps)
        self.dx = 0.15 / size
        self.dt = self.dx / (2 * self.c)
        self.pulse_width = 25 * size / 50
        self.default_steps = math.ceil((111 if self.pml else 210) * size / 50)
        self.center = size // 2

    def source(self, step):
        u = 4 * (step / self.pulse_width - 1)
        return np.float32(2 * u * math.exp(-u * u) / self.dx)

    def coefficients(self):
        n = self.size
        # qe = (ax, bx, ay, by), qh = (aHx, bHx, aHy, bHy).
        qe = np.empty((n + 1, n + 1, 4), dtype=np.float32)
        qh = np.empty_like(qe)
        qe[..., (0, 2)] = qh[..., (0, 2)] = 1
        qe[..., (1, 3)] = self.dt / self.eps / self.dx
        qh[..., (1, 3)] = self.dt / self.mu / self.dx
        p = self.pml
        if p:
            m = np.arange(1, p + 1, dtype=np.float64)
            sigma_max = -3 * self.eps * self.c * np.log(1e-5) / (2 * self.dx * p)
            sigma = sigma_max * ((m - .5) / (p + .5))**2
            rho = sigma_max * (self.mu / self.eps) * (m / (p + .5))**2
            ca = np.exp(-sigma * self.dt / self.eps)
            cb = -(ca - 1) / sigma / self.dx
            da = np.exp(-rho * self.dt / self.mu)
            db = -(da - 1) / rho / self.dx
            # Match the placement of the split-field Berenger PML in TM_OPEN.m.
            for array, axis, channel, a, b, offset in (
                    (qe, 0, 0, ca, cb, 1), (qe, 1, 2, ca, cb, 1),
                    (qh, 1, 0, da, db, 0), (qh, 0, 2, da, db, 0)):
                for k in range(p):
                    for position, coefficient in ((offset + k, p - 1 - k), (n - p + k, k)):
                        index = [slice(1, n), slice(1, n)]
                        index[axis] = position
                        array[(*index, channel)] = a[coefficient]
                        array[(*index, channel + 1)] = b[coefficient]
        return qe, qh


class TMReference:
    """Small NumPy float32 oracle; used for tests, never for the GPU benchmark."""

    def __init__(self, config):
        self.config = config
        n = config.size
        self.e = np.zeros((n + 1, n + 1, 4), dtype=np.float32)
        self.h = np.zeros((n + 1, n + 1, 2), dtype=np.float32)
        self.qe, self.qh = config.coefficients()
        self.n = 0

    def step(self):
        c = self.config
        n = c.size
        self.n += 1
        region = np.s_[1:n, 1:n]
        xd = self.h[1:n, 1:n, 1] - self.h[:n - 1, 1:n, 1]
        yd = self.h[1:n, :n - 1, 0] - self.h[1:n, 1:n, 0]
        if c.pml:
            self.e[region + (1,)] = (self.qe[region + (0,)] * self.e[region + (1,)]
                                    + self.qe[region + (1,)] * xd)
            self.e[region + (2,)] = (self.qe[region + (2,)] * self.e[region + (2,)]
                                    + self.qe[region + (3,)] * yd)
            self.e[region + (0,)] = self.e[region + (1,)] + self.e[region + (2,)]
        else:
            self.e[region + (0,)] += self.qe[region + (1,)] * (xd + yd)
        self.e[c.center, c.center, 0] = c.source(self.n)
        self.h[:, :n, 0] = (self.qh[:, :n, 0] * self.h[:, :n, 0]
                           + self.qh[:, :n, 1] * (self.e[:, :n, 0] - self.e[:, 1:, 0]))
        self.h[:n, :, 1] = (self.qh[:n, :, 2] * self.h[:n, :, 1]
                           + self.qh[:n, :, 3] * (self.e[1:, :, 0] - self.e[:n, :, 0]))


## 4. Compute shaders GLSL: один вызов на узел поля

`layout(local_size_x=16, local_size_y=16)` задаёт 256 потоков в рабочей группе.
`gl_GlobalInvocationID.xy` даёт координаты texel `(j, i)`. Число групп равно
`ceil((N+1)/16)` по каждой оси. На сетке 1000×1000 хранятся 1001×1001 узел:
запускаются 63×63 группы (1008×1008 потоков), а лишние потоки завершаются.

Как и во Fragment Shader, входные поля читаются через `texelFetch`.
Выход записывается через `imageStore` в отдельную текстуру. Форматы и
упаковка E/H, формулы, порядок арифметики и `precise` сохранены одинаковыми.
Никакой общей памяти рабочих групп и объединения временных шагов здесь нет.

Один шаг содержит два dispatch:

1. Старые E/H → новые E и источник.
2. Новые E и старые H → новые H.

Между ними `ctx.memory_barrier` делает результаты `imageStore` видимыми
последующему `texelFetch` и упорядочивает следующие записи. Барьер внутри
одного GLSL workgroup не синхронизировал бы всю сетку, поэтому этапы разделены
на два dispatch. Для чтения/сброса текстур на CPU используется отдельный
`TEXTURE_UPDATE_BARRIER_BIT`. Входные и выходные текстуры чередуются (ping-pong).


In [ ]:
E_COMPUTE_SHADER = """#version 430
layout(local_size_x=16, local_size_y=16, local_size_z=1) in;
uniform sampler2D old_e;
uniform sampler2D old_h;
uniform sampler2D qe;
uniform int size;
uniform bool use_pml;
uniform float source;
layout(rgba32f, binding=0) writeonly uniform image2D output_e;
void main() {
    ivec2 p = ivec2(gl_GlobalInvocationID.xy); // (j, i), not a screen coordinate
    if (p.x > size || p.y > size) return; // Last workgroups can exceed the grid.
    if (p.x == 0 || p.y == 0 || p.x == size || p.y == size) {
        imageStore(output_e, p, vec4(0)); return;
    }
    vec4 e = texelFetch(old_e, p, 0);
    vec4 q = texelFetch(qe, p, 0);
    vec2 h = texelFetch(old_h, p, 0).rg;
    float xd = h.y - texelFetch(old_h, p - ivec2(0,1), 0).g;
    float yd = texelFetch(old_h, p - ivec2(1,0), 0).r - h.x;
    // Same arithmetic order and precise qualifiers as the fragment baseline.
    precise float ex = q.x * e.y + q.y * xd;
    precise float ey = q.z * e.z + q.w * yd;
    precise float ez = use_pml ? ex + ey : e.x + q.y * (xd + yd);
    if (all(equal(p, ivec2(size / 2)))) ez = source;
    imageStore(output_e, p, vec4(ez, use_pml ? ex : 0, use_pml ? ey : 0, 0));
}
"""

H_COMPUTE_SHADER = """#version 430
layout(local_size_x=16, local_size_y=16, local_size_z=1) in;
uniform sampler2D new_e;
uniform sampler2D old_h;
uniform sampler2D qh;
uniform int size;
layout(rg32f, binding=0) writeonly uniform image2D output_h;
void main() {
    ivec2 p = ivec2(gl_GlobalInvocationID.xy);
    if (p.x > size || p.y > size) return;
    vec2 h = texelFetch(old_h, p, 0).rg;
    vec4 q = texelFetch(qh, p, 0);
    float ez = texelFetch(new_e, p, 0).r;
    precise float hx = 0;
    precise float hy = 0;
    if (p.x < size)
        hx = q.x * h.x + q.y * (ez - texelFetch(new_e, p + ivec2(1,0), 0).r);
    if (p.y < size)
        hy = q.z * h.y + q.w * (texelFetch(new_e, p + ivec2(0,1), 0).r - ez);
    imageStore(output_h, p, vec4(hx, hy, 0, 0));
}
"""


In [ ]:
class ComputeTM:
    """Two compute dispatches per step; sampler reads and imageStore writes.

    16x16 workgroups, no shared memory and no fused temporal steps. All fields
    and coefficients use the same textures as FragmentTM, for a comparable base.
    """

    workgroup_size = (16, 16, 1)

    def __init__(self, ctx, config):
        import moderngl
        self.ctx, self.config = ctx, config
        self.front, self.n = 0, 0
        n = config.size + 1
        if n > ctx.info["GL_MAX_TEXTURE_SIZE"]:
            raise ValueError("Grid exceeds GL_MAX_TEXTURE_SIZE")
        self.groups = ((n + 15) // 16, (n + 15) // 16, 1)
        self.shader_barriers = (moderngl.TEXTURE_FETCH_BARRIER_BIT
                                | moderngl.SHADER_IMAGE_ACCESS_BARRIER_BIT)
        self.texture_update_barrier = moderngl.TEXTURE_UPDATE_BARRIER_BIT
        self.resources = []
        try:
            def texture(channels, values=None):
                if values is None:
                    values = np.zeros((n, n, channels), dtype=np.float32)
                tex = ctx.texture((n, n), channels, np.ascontiguousarray(values).tobytes(), dtype="f4")
                self.resources.append(tex)
                tex.filter = (moderngl.NEAREST, moderngl.NEAREST)
                tex.repeat_x = tex.repeat_y = False
                return tex

            qe, qh = config.coefficients()
            self.qe, self.qh = texture(4, qe), texture(4, qh)
            self.e = [texture(4), texture(4)]
            self.h = [texture(2), texture(2)]
            self.eprog = ctx.compute_shader(E_COMPUTE_SHADER)
            self.resources.append(self.eprog)
            self.hprog = ctx.compute_shader(H_COMPUTE_SHADER)
            self.resources.append(self.hprog)
            self.eprog["old_e"].value, self.eprog["old_h"].value, self.eprog["qe"].value = 0, 1, 2
            self.hprog["new_e"].value, self.hprog["old_h"].value, self.hprog["qh"].value = 0, 1, 2
            self.eprog["size"].value = self.hprog["size"].value = config.size
            self.eprog["use_pml"].value = bool(config.pml)
            ctx.disable(moderngl.DEPTH_TEST | moderngl.BLEND | moderngl.CULL_FACE)
        except BaseException:
            self.release()
            raise

    def step(self):
        self.n += 1
        back = 1 - self.front
        self.e[self.front].use(0)
        self.h[self.front].use(1)
        self.qe.use(2)
        self.e[back].bind_to_image(0, read=False, write=True)
        self.eprog["source"].value = float(self.config.source(self.n))
        self.eprog.run(*self.groups)
        # imageStore is not automatically visible to later sampler reads.
        self.ctx.memory_barrier(self.shader_barriers)
        self.e[back].use(0)
        self.qh.use(2)
        self.h[back].bind_to_image(0, read=False, write=True)
        self.hprog.run(*self.groups)
        # Publish H to the next E pass, and E to the visualization sampler.
        # Also order later image writes after earlier shader memory accesses.
        self.ctx.memory_barrier(self.shader_barriers)
        self.front = back

    def read_fields(self):
        """Make image writes visible to glGetTexImage before GPU -> CPU transfer."""
        self.ctx.memory_barrier(self.texture_update_barrier)
        n = self.config.size + 1
        e = np.frombuffer(self.e[self.front].read(), np.float32).reshape(n, n, 4).copy()
        h = np.frombuffer(self.h[self.front].read(), np.float32).reshape(n, n, 2).copy()
        return e, h

    def reset(self):
        self.ctx.memory_barrier(self.texture_update_barrier)
        n = self.config.size + 1
        for tex in self.e:
            tex.write(bytes(n * n * 4 * 4))
        for tex in self.h:
            tex.write(bytes(n * n * 2 * 4))
        self.front, self.n = 0, 0

    def release(self):
        for resource in reversed(self.resources):
            resource.release()
        self.resources.clear()


## 5. Проверки корректности на GPU

Проверяем случайные начальные поля и полные траектории относительно NumPy
float32, обе границы, симметрию, поглощение и сброс состояния. Дополнительный
тест проверяет сетки, заканчивающиеся внутри рабочей группы 16×16.
Допуск нормированной максимальной ошибки компоненты — 5×10⁻⁵;
побитовое совпадение не требуется.


In [ ]:
import unittest
class ComputeTests(unittest.TestCase):
    context = None

    @classmethod
    def setUpClass(cls):
        if cls.context is None:
            raise unittest.SkipTest("An NVIDIA EGL context is required")

    def assert_fields_close(self, gpu, reference):
        e, h = gpu.read_fields()
        self.assertEqual(e.dtype, np.dtype("float32"))
        self.assertEqual(h.dtype, np.dtype("float32"))
        for actual, expected in ((e, reference.e), (h, reference.h)):
            for channel in range(actual.shape[-1]):
                scale = max(float(np.max(np.abs(expected[..., channel]))), 1e-8)
                relative_max_error = float(np.max(np.abs(actual[..., channel] - expected[..., channel]))) / scale
                self.assertLess(relative_max_error, 5e-5)
        self.assertTrue(np.all(e[[0, -1]] == 0))
        self.assertTrue(np.all(e[:, [0, -1]] == 0))

    def test_random_stencil_both_boundaries(self):
        rng = np.random.default_rng(19)
        for boundary in ("pec", "pml"):
            with self.subTest(boundary=boundary):
                config = TMConfig(32, boundary)
                reference = TMReference(config)
                reference.e[:] = rng.normal(size=reference.e.shape).astype(np.float32)
                reference.e[..., 3] = 0
                if boundary == "pec":
                    reference.e[..., 1:3] = 0
                reference.e[[0, -1]] = 0
                reference.e[:, [0, -1]] = 0
                reference.h[:] = rng.normal(scale=.01, size=reference.h.shape).astype(np.float32)
                reference.h[:, -1, 0] = reference.h[-1, :, 1] = 0
                gpu = ComputeTM(self.context, config)
                try:
                    gpu.e[0].write(reference.e.tobytes())
                    gpu.h[0].write(reference.h.tobytes())
                    gpu.step()
                    reference.step()
                    self.assert_fields_close(gpu, reference)
                finally:
                    gpu.release()

    def test_complete_pulse_trajectories(self):
        for boundary in ("pec", "pml"):
            with self.subTest(boundary=boundary):
                config = TMConfig(50, boundary)
                reference = TMReference(config)
                gpu = ComputeTM(self.context, config)
                try:
                    for step in range(config.default_steps):
                        reference.step()
                        gpu.step()
                        if (step + 1) % 37 == 0:
                            self.assert_fields_close(gpu, reference)
                    self.assert_fields_close(gpu, reference)
                finally:
                    gpu.release()

    def test_symmetry_and_absorption(self):
        fields = {}
        for boundary in ("pec", "pml"):
            gpu = ComputeTM(self.context, TMConfig(50, boundary))
            try:
                for _ in range(210):
                    gpu.step()
                e, h = gpu.read_fields()
                self.assertTrue(np.isfinite(e).all() and np.isfinite(h).all())
                fields[boundary] = e[..., 0]
            finally:
                gpu.release()
        np.testing.assert_allclose(fields["pec"], fields["pec"].T, atol=2e-4)
        np.testing.assert_allclose(fields["pec"], fields["pec"][::-1], atol=2e-4)
        core = np.s_[8:-8, 8:-8]
        self.assertLess(np.linalg.norm(fields["pml"][core]), .1 * np.linalg.norm(fields["pec"][core]))

    def test_reset_reproducibility(self):
        gpu = ComputeTM(self.context, TMConfig(32))
        try:
            for _ in range(20):
                gpu.step()
            expected = gpu.read_fields()
            gpu.reset()
            self.assertEqual(gpu.n, 0)
            self.assertTrue(all(np.all(array == 0) for array in gpu.read_fields()))
            for _ in range(20):
                gpu.step()
            for actual, initial in zip(gpu.read_fields(), expected):
                np.testing.assert_array_equal(actual, initial)
        finally:
            gpu.release()

    def test_partial_workgroups(self):
        # (N+1) texels can end inside a 16x16 workgroup; check both axes/edges.
        for size in (16, 17, 31, 32, 33):
            for boundary in ("pec", "pml"):
                with self.subTest(size=size, boundary=boundary):
                    config = TMConfig(size, boundary)
                    reference = TMReference(config)
                    gpu = ComputeTM(self.context, config)
                    try:
                        for _ in range(12):
                            gpu.step()
                            reference.step()
                        self.assert_fields_close(gpu, reference)
                    finally:
                        gpu.release()


In [ ]:
# @title Запуск GPU-тестов
import sys
ComputeTests.context = ctx
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ComputeTests)
)
if not result.wasSuccessful() or result.testsRun != 5 or result.skipped:
    raise AssertionError("GPU-тесты должны пройти до замеров и создания видео")


## 6. Параметры эксперимента

При сетке 1000×1000 и PML исходные 111 шагов превращаются в 2220 шагов:
уменьшение клетки требует уменьшения шага времени. Это около 0,555 нс.
`VIDEO_STEPS=0` выбирает такую исходную физическую длительность.
FPS влияет только на воспроизведение, `FRAME_STRIDE` — на частоту съёмки.


In [ ]:
# @title Параметры
GRID_SIZE = 1000 # @param {type:"integer"}
BOUNDARY = "pml" # @param ["pml", "pec"]
BENCHMARK_STEPS = 300 # @param {type:"integer"}
WARMUP_STEPS = 300 # @param {type:"integer"}
REPEATS = 3 # @param {type:"integer"}
VIDEO_STEPS = 0 # @param {type:"integer"}
FRAME_STRIDE = 20 # @param {type:"integer"}
FPS = 25 # @param {type:"integer"}
FIELD_LIMIT = 1000.0 # @param {type:"number"}
for name in ("GRID_SIZE", "BENCHMARK_STEPS", "WARMUP_STEPS", "REPEATS", "VIDEO_STEPS", "FRAME_STRIDE", "FPS"):
    value = globals()[name]
    if not isinstance(value, int) or isinstance(value, bool):
        raise ValueError(f"{name} должен быть целым числом")
if min(BENCHMARK_STEPS, WARMUP_STEPS, REPEATS, FRAME_STRIDE, FPS) < 1 or VIDEO_STEPS < 0:
    raise ValueError("Некорректные параметры")
if not math.isfinite(FIELD_LIMIT) or FIELD_LIMIT <= 0:
    raise ValueError("FIELD_LIMIT должен быть положительным конечным числом")
config = TMConfig(GRID_SIZE, BOUNDARY)
video_steps = VIDEO_STEPS or config.default_steps
if video_steps < FRAME_STRIDE:
    raise ValueError("Число шагов должно быть не меньше FRAME_STRIDE")
print(f"{GRID_SIZE}×{GRID_SIZE}, dx={config.dx:.6g} м, dt={config.dt:.6g} с")
print(f"Рабочие группы: {(GRID_SIZE + 16) // 16}×{(GRID_SIZE + 16) // 16}, по 16×16 потоков")
print(f"Видео: {video_steps} шагов, {video_steps // FRAME_STRIDE} кадров, "
      f"физическое время {video_steps * config.dt * 1e9:.3f} нс")


## 7. Замер расчёта без видео

Инициализация, компиляция шейдеров, прогрев и чтение полей исключены из замера.
Каждый повтор начинается с нулевых полей. Сохраняются GPU timer query
и полное синхронизированное время цикла Python→OpenGL.

**GPU timer query охватывает всю серию проходов** и может включать ожидание
GPU, пока Python отправляет очередные команды. Поэтому это время данного
способа запуска решателя, а не изолированная пропускная способность GPU.
Сравнивать эти числа с отдельным прошлым запуском Fragment Shader нельзя:
для вывода о скорости нужны оба решателя в одном контексте и одинаковые замеры.


In [ ]:
def benchmark_compute(ctx, config, steps=300, repeats=3, warmup=300):
    """GPU timer query + synchronized wall time, excluding allocation and readback."""
    if steps < 1 or repeats < 1 or warmup < 1:
        raise ValueError("steps, repeats and warmup must be positive")
    sim = ComputeTM(ctx, config)
    results = []
    try:
        for _ in range(warmup):
            sim.step()
        ctx.finish()
        for repeat in range(repeats):
            sim.reset()
            ctx.finish()
            query = ctx.query(time=True)
            start = time.perf_counter()
            with query:
                for _ in range(steps):
                    sim.step()
            ctx.finish()
            wall_seconds = time.perf_counter() - start
            gpu_seconds = query.elapsed / 1e9
            if gpu_seconds <= 0:
                raise RuntimeError("GPU timer returned no valid duration")
            results.append(dict(repeat=repeat + 1, gpu_seconds=gpu_seconds,
                                wall_seconds=wall_seconds,
                                gpu_ms_per_step=1000 * gpu_seconds / steps,
                                wall_ms_per_step=1000 * wall_seconds / steps,
                                million_cell_steps_per_second=config.size**2 * steps / gpu_seconds / 1e6))
        return dict(backend="compute", workgroup_size=list(sim.workgroup_size),
                    size=config.size, dtype="float32", boundary=config.boundary,
                    steps=steps, warmup=warmup, repeats=results,
                    note="GPU query spans the submitted passes and may include GPU idle gaps "
                         "while Python submits commands; wall time includes host dispatch.")
    finally:
        sim.release()


In [ ]:
# @title GPU-замеры
benchmark = benchmark_compute(ctx, config, steps=BENCHMARK_STEPS,
                               repeats=REPEATS, warmup=WARMUP_STEPS)
benchmark["opengl"] = {key: ctx.info[key] for key in ("GL_VENDOR", "GL_RENDERER", "GL_VERSION")}
for row in benchmark["repeats"]:
    print(f"Повтор {row['repeat']}: GPU interval {row['gpu_ms_per_step']:.3f} мс/шаг; "
          f"wall {row['wall_ms_per_step']:.3f} мс/шаг; "
          f"{row['million_cell_steps_per_second']:.1f} млн ячеек·шаг/с")
_ = (WORK_DIR / "benchmark.json").write_text(json.dumps(benchmark, indent=2))


## 8. Видео под ячейкой

Решатель запускается заново. Отдельный **fragment shader** раскрашивает Ez:
**синий — отрицательное поле, белый — ноль, красный — положительное**.
Физические поля не копируются на CPU на каждом шаге: передаются только RGB-кадры
по `FRAME_STRIDE`. CPU кодирует их в MP4/H.264. Эта работа не входит в замер выше.
В конце сохраняются финальные поля NPZ.


In [ ]:
VERTEX_SHADER = """#version 430
// One full-screen triangle; no vertex buffer and no diagonal between triangles.
void main() {
    vec2 positions[3] = vec2[](vec2(-1,-1), vec2(3,-1), vec2(-1,3));
    gl_Position = vec4(positions[gl_VertexID], 0, 1);
}
"""

COLOR_SHADER = """#version 430
uniform sampler2D field;
uniform float limit;
uniform vec2 output_size;
layout(location=0) out vec4 color;
void main() {
    // Blue = negative, white = zero, red = positive. GL row 0 is the lower row.
    float value = texture(field, gl_FragCoord.xy / output_size).r;
    float t = clamp(value / limit, -1.0, 1.0);
    vec3 rgb = t < 0 ? mix(vec3(1), vec3(0.1,0.25,0.9), -t)
                     : mix(vec3(1), vec3(0.9,0.15,0.1), t);
    color = vec4(rgb, 1);
}
"""


In [ ]:
def render_compute_video(ctx, config, output, steps, stride=20, fps=25, resolution=800, limit=50):
    """GPU color pass -> RGB readback -> FFmpeg; independent of the solver benchmark."""
    import moderngl
    import subprocess
    if steps < stride or stride < 1 or fps < 1 or resolution < 2 or resolution % 2 or limit <= 0:
        raise ValueError("Invalid video settings; resolution must be even")
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    sim = ComputeTM(ctx, config)
    resources = []
    process = None
    frame_count = 0
    start = time.perf_counter()
    try:
        rgb = ctx.texture((resolution, resolution), 4, dtype="f1")
        resources.append(rgb)
        fbo = ctx.framebuffer([rgb])
        resources.append(fbo)
        program = ctx.program(vertex_shader=VERTEX_SHADER, fragment_shader=COLOR_SHADER)
        resources.append(program)
        vao = ctx.vertex_array(program, [])
        resources.append(vao)
        program["field"].value = 0
        program["limit"].value = float(limit)
        program["output_size"].value = (resolution, resolution)
        # Raw OpenGL rows are bottom-up; vflip restores the displayed orientation.
        command = ["ffmpeg", "-y", "-v", "error", "-f", "rawvideo", "-pixel_format", "rgb24",
                   "-video_size", f"{resolution}x{resolution}", "-framerate", str(fps),
                   "-i", "pipe:0", "-vf", "vflip", "-an", "-c:v", "libx264", "-preset", "fast",
                   "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(output)]
        # Log to disk rather than a stderr pipe that can block during encoding.
        with output.with_suffix(".ffmpeg.log").open("wb") as log:
            process = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=log)
            try:
                for _ in range(steps):
                    sim.step()
                    if sim.n % stride == 0:
                        fbo.use()
                        sim.e[sim.front].use(0)
                        vao.render(vertices=3)
                        process.stdin.write(fbo.read(components=3, alignment=1))
                        frame_count += 1
            finally:
                process.stdin.close()
                returncode = process.wait()
            if returncode:
                raise RuntimeError(f"FFmpeg failed ({returncode}); see {output.with_suffix('.ffmpeg.log')}")
        e, h = sim.read_fields()
        np.savez_compressed(output.with_suffix(".npz"), ez=e[..., 0], ezx=e[..., 1], ezy=e[..., 2],
                            hx=h[:, :-1, 0], hy=h[:-1, :, 1], step=sim.n,
                            dx=config.dx, dt=config.dt, size=config.size, boundary=config.boundary)
        return dict(frames=frame_count, fps=fps, steps=steps, size=config.size,
                    resolution=resolution, wall_seconds=time.perf_counter() - start,
                    maximum_field=float(np.max(np.abs(e[..., 0]))), video_bytes=output.stat().st_size)
    finally:
        for resource in reversed(resources):
            resource.release()
        sim.release()


In [ ]:
# @title Расчёт и показ видео
from IPython.display import Video, display
video_path = WORK_DIR / "compute_tm.mp4"
video_report = render_compute_video(
    ctx, config, video_path, steps=video_steps, stride=FRAME_STRIDE,
    fps=FPS, resolution=800, limit=FIELD_LIMIT,
)
print(json.dumps(video_report, indent=2))
display(Video(str(video_path), embed=True, mimetype="video/mp4", width=720,
              html_attributes="controls loop playsinline"))


In [ ]:
# @title Проверка MP4 и численных полей
import subprocess
from fractions import Fraction
probe = json.loads(subprocess.check_output([
    "ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
    "-show_entries", "stream=codec_name,width,height,r_frame_rate,nb_read_frames",
    "-of", "json", str(video_path),
], text=True))["streams"][0]
assert probe["codec_name"] == "h264"
assert int(probe["nb_read_frames"]) == video_report["frames"] == video_steps // FRAME_STRIDE
assert Fraction(probe["r_frame_rate"]) == FPS
assert (probe["width"], probe["height"]) == (800, 800)
with np.load(video_path.with_suffix(".npz")) as data:
    for name in ("ez", "ezx", "ezy", "hx", "hy"):
        assert data[name].dtype == np.dtype("float32")
        assert np.isfinite(data[name]).all()
    assert data["ez"].shape == (GRID_SIZE + 1, GRID_SIZE + 1)
    assert int(data["step"]) == video_steps
report = dict(opengl=benchmark["opengl"], tests_run=result.testsRun,
              tests_ok=result.wasSuccessful(), video=video_report, ffprobe=probe)
_ = (WORK_DIR / "verification.json").write_text(json.dumps(report, indent=2))
print("Видео и float32-поля проверены")


In [ ]:
# @title 9. Архив результатов
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
import zipfile
archive_path = WORK_DIR / "compute_results.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("compute_tm.mp4", "compute_tm.npz", "benchmark.json",
                 "verification.json", "opengl_context.json"):
        archive.write(WORK_DIR / name, arcname=name)
print(f"Архив: {archive_path} ({archive_path.stat().st_size / 1024:.0f} КиБ)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))
print("COMPUTE_NOTEBOOK_OK")
